# AutonomousPlanningAgent and the Agent Framework

## Order

1. Modal.com and SpecialistAgent  
2. RAG, FrontierAgent, EnsembleAgent  
3. ScannerAgent, MessagingAgent  
4. AutonomousPlanningAgent and WineAgentFramework  
5. Gradio finale

Now it's time for the planning layer. Two flavors exist:

- `PlanningAgent` — deterministic Python orchestration (scan, judge each wine
  with the ensemble, alert the best if it clears a threshold). Used by the
  `WineAgentFramework` for unattended runs.
- `AutonomousPlanningAgent` — the same three capabilities exposed as **tools**
  to a frontier LLM, which decides for itself what to call and when.

This notebook builds up to the autonomous one: first with fake tools to see
the agentic loop mechanics, then the real thing.

In [ ]:
import json
import logging

import chromadb
from dotenv import load_dotenv
from openai import OpenAI

from agents.scanner_agent import ScannerAgent

load_dotenv(override=True)
openai_client = OpenAI()
MODEL = "gpt-5.1"

## Start with some test data

Hardcoded wines from a real scan — so we can exercise the loop without
scraping or paying for scanner calls.

In [ ]:
test_results = ScannerAgent().test_scan()
test_results

## Three pretend functions

The autonomous agent's tools, faked: scan returns the test data (with each
wine's actual VFM at the shop price), estimate always answers 55, notify just
prints.

In [ ]:
def scan_the_wine_shop_for_bargains() -> str:
    """Fake scan — returns the hardcoded selection with actual VFM attached"""
    print("Fake function to scan the wine shop - returns hardcoded wines")
    enriched = [
        {**listing.model_dump(), "actual_vfm": listing.actual_vfm()}
        for listing in test_results.listings
    ]
    return json.dumps({"listings": enriched})

In [ ]:
def estimate_typical_vfm(url: str) -> str:
    """Fake estimate — always answers VFM 55"""
    print(f"Fake function estimating typical VFM of {url[:60]}... - always 55")
    return f"The tasting profile of the wine at {url} typically delivers VFM 55"

In [ ]:
def notify_user_of_bargain(url: str, estimated_vfm: int) -> str:
    """Fake notify — just prints what would be pushed"""
    print(f"Fake function notifying user of {url} with estimated VFM {estimated_vfm}")
    return "Notification sent ok"

### Try one out

In [ ]:
notify_user_of_bargain("https://bottlebarn.com/products/2024-bogle-sauvignon-blanc", 55)

### The tool schemas — a big block of JSON

This is what the LLM actually sees: each tool's name, what it does, and what
arguments it takes. Note the wines travel between tools **by URL** — the code
resolves the URL back to the scanned listing, so the ensemble always receives
the deterministic summary, never an LLM paraphrase.

In [ ]:
scan_function = {
    "name": "scan_the_wine_shop_for_bargains",
    "description": (
        "Returns the best-value wine listings from the shop, each with its printed "
        "critic score, shop price, URL, and the actual VFM (0-99) at that price"
    ),
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
}

estimate_function = {
    "name": "estimate_typical_vfm",
    "description": (
        "Given a listing's URL, estimate the VFM (0-99) that this wine's tasting "
        "profile typically delivers, judged from its description alone"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the listing, exactly as returned by the scan",
            },
        },
        "required": ["url"],
        "additionalProperties": False,
    },
}

notify_function = {
    "name": "notify_user_of_bargain",
    "description": (
        "Send the user a push notification about the single best bargain; "
        "only call this one time"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the chosen listing, exactly as scanned",
            },
            "estimated_vfm": {
                "type": "integer",
                "description": "The typical VFM you obtained from the estimate tool",
            },
        },
        "required": ["url", "estimated_vfm"],
        "additionalProperties": False,
    },
}

In [ ]:
tools = [
    {"type": "function", "function": scan_function},
    {"type": "function", "function": estimate_function},
    {"type": "function", "function": notify_function},
]
tools

In [ ]:
def handle_tool_call(message):
    """Actually call the tools the model asked for"""
    results = []
    for tool_call in message.tool_calls:
        print("MESSAGE\n", message)
        tool_name = tool_call.function.name
        print("TOOL NAME\n", tool_name)
        arguments = json.loads(tool_call.function.arguments)
        print("ARGUMENTS\n", arguments)
        tool = globals().get(tool_name)
        print("TOOL\n", tool)
        result = tool(**arguments) if tool else ""
        print("RESULT\n", result)
        results.append(
            {"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id}
        )
    return results

In [ ]:
system_message = (
    "You find great value wines using your tools, and notify the user of the "
    "single best bargain."
)
user_message = """First, use your tool to scan the wine shop for promising listings — each
comes with its actual VFM (value for money, 0-99) at the shop price. Then for each listing,
use your tool to estimate the VFM its tasting profile typically delivers. Then pick the
single wine whose actual VFM most exceeds its typical estimate, and use your tool to notify
the user, passing that wine's url and your estimated typical VFM. Then just reply OK to
indicate success."""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

### The agentic loop

Keep calling the model; whenever it asks for tools, run them and feed the
results back; stop when it answers in plain text. Watch the fake functions
print as the model works through scan → estimate ×3 → notify.

In [ ]:
done = False
while not done:
    response = openai_client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True

response.choices[0].message.content

In [ ]:
messages

## And now... the real Autonomous Planning Agent

Same loop, real tools: live scan, real ensemble estimates (each one is a
frontier call + a Modal call + the local network), real notification.

**Cost/time note:** one `plan()` run makes ~5 ensemble estimates plus the
planning loop itself — some tens of cents, a few minutes if the Modal
container needs to wake.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
DB = "wine_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("wines")

In [ ]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent

agent = AutonomousPlanningAgent(collection)

In [ ]:
agent.plan()